# NLP 数据质量流水线：从原始字节到可回放数据集

这份 notebook 把“清洗文本”改写成一份可执行的数据合同。目标不是把字符串洗得越短越好，而是让每条样本都能回答：来自哪里、用哪版规则处理、哪些内容被拒绝、怎样复现同一训练集。

完整链路：

原始字节不可变存档 → schema/编码检查 → 安全规范化视图 → PII 日志脱敏 → exact/near dedup → 文档族隔离切分 → 质量门禁 → 漂移监控 → manifest 回放

受控样本只验证实现与边界，不代表真实语料质量。

## 1. 业务目标与外部合同

输入记录至少包含 doc_id、tenant_id、source、created_at、declared_encoding、payload_bytes。输出不覆盖原文，而是新增 decoded_text、normalized_text、clean_to_raw、pii_types、family_id、split、rule_version、decision/reasons。

三个不变量：

1. E1042、HTTP 500、金额等业务标识不能被清洗规则破坏。
2. “不、未、禁止、不能”等否定词不能当停用词删除，否则含义反转。
3. 训练 offset 必须指向不可变 raw text；若展示规范化视图，必须携带映射或重新标注。

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
import codecs
import hashlib
import html
import json
import math
import re
import unicodedata
from datetime import datetime, timezone

RULE_VERSION = 'quality-v1.2.0'
SCHEMA_VERSION = 'raw-doc-v1'

@dataclass(frozen=True)
class RawRecord:
    doc_id: str
    tenant_id: str
    source: str
    created_at: str
    declared_encoding: str
    payload_bytes: bytes

def b(text, encoding='utf-8'):
    return text.encode(encoding)

records = [
    RawRecord('a-001', 'tenant-a', 'ticket', '2026-07-01T10:00:00Z', 'utf-8',
              b('错误码 Ｅ１０４２：数据库不能连接。联系 alice@example.com，电话 13800138000。')),
    RawRecord('a-002', 'tenant-a', 'ticket', '2026-07-01T10:03:00Z', 'utf-8',
              b('错误码 E1042：数据库不能连接。 联系 alice@example.com，电话 13800138000。')),
    RawRecord('a-003', 'tenant-a', 'manual', '2026-07-02T09:00:00Z', 'utf-8',
              b('退款未到账时，不要重复提交；保留订单号 ORD-7788。')),
    RawRecord('a-004', 'tenant-a', 'manual', '2026-07-02T09:05:00Z', 'gb18030',
              b('退款未到账时，不要重复提交；保留订单号 ORD-7788。', 'gb18030')),
    RawRecord('a-005', 'tenant-a', 'chat', '2026-07-03T09:00:00Z', 'utf-8',
              b('用户说：HTTP 500 仍未恢复，邮箱 bob@test.cn。')),
    RawRecord('b-001', 'tenant-b', 'private', '2026-07-04T09:00:00Z', 'utf-8',
              b('租户 B 的内部密钥不能外发。')),
    RawRecord('bad-001', 'tenant-a', 'chat', 'not-a-time', 'utf-8', b('短')),
]
assert len({r.doc_id for r in records}) == len(records)
print('受控记录数:', len(records))

## 2. Schema、来源与泄漏

字段完整不等于可训练。重复工单、同一网页的打印版、同一会话切片属于一个文档族；若先随机切分再去重，近重复内容会跨越 train/test，指标会虚高。正确顺序是先建立 family，再按 family 切分。

created_at 必须严格解析为带时区的时间，并在进入排序/窗口逻辑前统一到 UTC；仅能被解析但没有时区的 `2026-07-01 10:00` 仍是不合格输入。tenant 必须来自受信身份或采集配置，而不是从正文猜测。原始 payload 只读保存，后续规则只能生成派生列。

In [ ]:
DOC_ID_RE = re.compile(r'^[a-z0-9][a-z0-9._-]{2,63}$')
ALLOWED_ENCODINGS = {'utf-8', 'utf-8-sig', 'gb18030'}

def validate_schema(record):
    reasons = []
    if not DOC_ID_RE.fullmatch(record.doc_id):
        reasons.append('invalid_doc_id')
    if not record.tenant_id:
        reasons.append('missing_tenant')
    if record.declared_encoding.lower() not in ALLOWED_ENCODINGS:
        reasons.append('encoding_not_allowed')
    if not isinstance(record.payload_bytes, bytes) or not record.payload_bytes:
        reasons.append('empty_payload')
    try:
        parsed_time = datetime.fromisoformat(record.created_at.replace('Z', '+00:00'))
        if parsed_time.tzinfo is None or parsed_time.utcoffset() is None:
            reasons.append('timezone_required')
        else:
            parsed_time.astimezone(timezone.utc)
    except (ValueError, TypeError):
        reasons.append('invalid_created_at')
    return reasons

schema_results = {r.doc_id: validate_schema(r) for r in records}
naive_time_record = RawRecord('naive-001', 'tenant-a', 'fixture', '2026-07-01 10:00:00',
                              'utf-8', b('带时区合同测试'))
assert schema_results['a-001'] == []
assert schema_results['bad-001'] == ['invalid_created_at']
assert validate_schema(naive_time_record) == ['timezone_required']
print(schema_results)

## 3. 编码与语言识别要显式失败

生产中优先相信带 BOM/协议元数据的编码，再用允许列表严格解码；不要用 errors=ignore 静默丢字节。编码探测只能提供候选和置信度，低置信度进入隔离队列。

语言识别同样不是单一标签真理。短文本、代码、专有名词常低置信；这里实现的汉字/ASCII 比例仅用于路由示范，不能替代经过评估的语言识别模型。

In [ ]:
def strict_decode(payload, declared):
    enc = declared.lower()
    if payload.startswith(codecs.BOM_UTF8):
        enc = 'utf-8-sig'
    if enc not in ALLOWED_ENCODINGS:
        raise ValueError('encoding_not_allowed')
    return payload.decode(enc, errors='strict'), enc

def coarse_language(text):
    han = sum('\u4e00' <= ch <= '\u9fff' for ch in text)
    latin = sum(ch.isascii() and ch.isalpha() for ch in text)
    letters = han + latin
    if letters < 4:
        return {'label': 'unknown', 'confidence': 0.0}
    ratio = han / letters
    label = 'zh' if ratio >= 0.55 else ('en' if ratio <= 0.2 else 'mixed')
    return {'label': label, 'confidence': round(max(ratio, 1-ratio), 3)}

decoded = {}
for r in records:
    decoded[r.doc_id] = strict_decode(r.payload_bytes, r.declared_encoding)[0]
assert decoded['a-003'] == decoded['a-004']
# 邮箱中的拉丁字符会让这条规则基线判为 mixed；这是需要分桶而非伪装高置信的信号。
assert coarse_language(decoded['a-001'])['label'] == 'mixed'
assert coarse_language('E1042') ['label'] == 'unknown'
print(coarse_language(decoded['a-005']))

## 4. Unicode 与安全清洗：保留 raw，另建检索视图

NFC 适合规范等价字符；NFKC 还会折叠兼容字符，例如全角字母。是否使用 NFKC 是业务决定：搜索视图可能需要它，法律原文和密码字段通常不能用它覆盖原值。

下面先计算整串 NFKC，再建立 clean_to_raw 映射。若组合字符或 Hangul 等发生跨 code point 合成，简单的一对多索引不足以表达来源范围，函数会显式拒绝并要求使用支持 provenance range 的规范化器或重新标注，而不是返回看似可用的错误 offset。兼容字符展开产生的重复 raw index 也有边界检查，不能只截取展开结果的一半再投影。HTML 解码、大小写、标点删除必须分别版本化。

In [ ]:
PROTECTED_PATTERNS = [
    re.compile(r'\b[A-Z]{1,8}[-_]?\d{2,10}\b'),
    re.compile(r'\bHTTP\s+\d{3}\b', re.I),
]
NEGATIONS = {'不', '未', '没有', '禁止', '不能', '不要'}

def normalize_view(raw):
    # 先验证逐字符来源能否忠实解释整串 NFKC；跨字符合成必须交给更强的 range mapper。
    pieces, origins = [], []
    for raw_index, ch in enumerate(raw):
        piece = unicodedata.normalize('NFKC', ch)
        pieces.append(piece)
        origins.extend([raw_index] * len(piece))
    whole_nfkc = unicodedata.normalize('NFKC', raw)
    if ''.join(pieces) != whole_nfkc:
        raise ValueError('non_local_normalization_requires_range_mapping')

    out, clean_to_raw = [], []
    previous_space = False
    for normalized_ch, raw_index in zip(whole_nfkc, origins):
        if unicodedata.category(normalized_ch) in {'Cc', 'Cf'} and normalized_ch not in '\n\t':
            continue
        if normalized_ch.isspace():
            if previous_space:
                continue
            normalized_ch = ' '
            previous_space = True
        else:
            previous_space = False
        out.append(normalized_ch)
        clean_to_raw.append(raw_index)
    joined = ''.join(out)
    normalized = joined.strip()
    if joined != normalized:
        left_trim = len(joined) - len(joined.lstrip())
        clean_to_raw = clean_to_raw[left_trim:left_trim + len(normalized)]
    return normalized, clean_to_raw

norm1, map1 = normalize_view(decoded['a-001'])
assert 'E1042' in norm1
assert '不能' in norm1
assert len(norm1) == len(map1)
pos = norm1.index('E1042')
assert decoded['a-001'][map1[pos]] == 'Ｅ'
assert unicodedata.normalize('NFKC', 'e\u0301') == 'é'
print(norm1)

### 4.1 失败反例：删除标点再复用旧 offset

若原文是“北京，海淀”，删除逗号后“海淀”的起点从 3 变成 2。直接把规范化文本上的 span 写回 raw 会错一位。更危险的是删除“未/不”，会把负例改成正例。

工程策略：raw_text 与 raw_span 是事实层；normalized_text 是特征层；任何跨层 span 都必须显式投影，并在无法一一映射时拒绝自动转换。

In [ ]:
def project_clean_span(clean_to_raw, start, end):
    if not (0 <= start < end <= len(clean_to_raw)):
        raise ValueError('invalid_clean_span')
    # 同一 raw 字符可能展开成多个 clean 字符；span 不得切在展开结果内部。
    if start > 0 and clean_to_raw[start] == clean_to_raw[start - 1]:
        raise ValueError('span_starts_inside_normalized_expansion')
    if end < len(clean_to_raw) and clean_to_raw[end - 1] == clean_to_raw[end]:
        raise ValueError('span_ends_inside_normalized_expansion')
    raw_positions = clean_to_raw[start:end]
    return min(raw_positions), max(raw_positions) + 1

clean, mapping = normalize_view('北京，海淀 E1042')
start = clean.index('海淀')
raw_start, raw_end = project_clean_span(mapping, start, start + 2)
assert '北京，海淀 E1042'[raw_start:raw_end] == '海淀'
expanded, expanded_map = normalize_view('ﬃ')
assert expanded == 'ffi' and project_clean_span(expanded_map, 0, 3) == (0, 1)
try:
    project_clean_span(expanded_map, 0, 1)
    raise AssertionError('不能把兼容字符展开结果的一部分投影成完整 raw span')
except ValueError as exc:
    assert 'inside_normalized_expansion' in str(exc)
try:
    normalize_view('e\u0301')
    raise AssertionError('跨 code point 合成必须显式拒绝简单索引映射')
except ValueError as exc:
    assert str(exc) == 'non_local_normalization_requires_range_mapping'
print({'clean_span': (start, start+2), 'raw_span': (raw_start, raw_end)})

## 5. PII：用于日志的等长脱敏，不等于删除源数据

邮箱和手机号可能需要基于用途、地区法规与授权处理。示例只演示确定性规则：输出等长 mask，保持字符位置，记录类型与规则版本；原文进入受控存储，不应出现在普通日志、trace 或错误堆栈。

规则有漏检和误杀风险，尤其是订单号、IPv6、姓名和地址。生产应叠加模型/词典、人工复核和数据保留策略。

In [ ]:
PII_RULES = {
    'email': re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b'),
    'cn_mobile': re.compile(r'(?<!\d)1[3-9]\d{9}(?!\d)'),
}

def same_length_mask(value):
    return ''.join(ch if ch in '@.-_ ' else '*' for ch in value)

def redact_for_log(text):
    hits = []
    redacted = text
    for pii_type, pattern in PII_RULES.items():
        matches = list(pattern.finditer(redacted))
        for match in reversed(matches):
            hits.append({'type': pii_type, 'start': match.start(), 'end': match.end()})
            redacted = redacted[:match.start()] + same_length_mask(match.group()) + redacted[match.end():]
    return redacted, sorted(hits, key=lambda x: x['start'])

masked, pii_hits = redact_for_log(decoded['a-001'])
assert len(masked) == len(decoded['a-001'])
assert 'alice@example.com' not in masked
assert 'Ｅ１０４２' in masked
assert masked.index('Ｅ１０４２') == decoded['a-001'].index('Ｅ１０４２')
assert {h['type'] for h in pii_hits} == {'email', 'cn_mobile'}
print(masked, pii_hits)

## 6. Exact dedup：先定义等价关系

byte hash 只能发现完全相同字节。面向内容的 exact key 可以基于规则化视图，但规则越激进，越可能把含义不同的文档误合并。本例只做 NFKC、空白折叠和小写；绝不删除否定词、数字或错误码。

dedup key 必须包含 tenant/安全域，避免跨租户通过共享 hash 暴露内容存在性。

In [ ]:
def canonical_for_dedup(text):
    normalized, _ = normalize_view(text)
    return normalized.casefold()

def exact_key(tenant_id, text):
    payload = (tenant_id + '\0' + canonical_for_dedup(text)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

exact_groups = defaultdict(list)
for r in records:
    if not validate_schema(r):
        exact_groups[exact_key(r.tenant_id, decoded[r.doc_id])].append(r.doc_id)

assert exact_key('tenant-a', decoded['a-003']) == exact_key('tenant-a', decoded['a-004'])
assert exact_key('tenant-a', '不能退款') != exact_key('tenant-a', '能退款')
assert exact_key('tenant-a', '相同') != exact_key('tenant-b', '相同')
print([ids for ids in exact_groups.values() if len(ids) > 1])

## 7. Near dedup：shingle + MinHash 候选，再做精确确认

字符 k-shingle 的 Jaccard 相似度能覆盖中文且不依赖分词。MinHash 利用多个随机哈希的最小值近似 Jaccard，可配合 LSH 减少全量两两比较。

教学数据很小，所以先展示 MinHash 签名，再用真实 Jaccard 确认合并。线上不可只凭一个短签名删除文档；阈值要按来源、长度和误合并成本标定，并保存代表文档与成员列表。

In [ ]:
def shingles(text, k=3):
    compact = re.sub(r'\s+', '', canonical_for_dedup(text))
    return {compact[i:i+k] for i in range(max(0, len(compact)-k+1))} or {compact}

def stable_hash(token, seed):
    return int.from_bytes(hashlib.blake2b(
        f'{seed}:{token}'.encode(), digest_size=8).digest(), 'big')

def minhash_signature(tokens, num_perm=32):
    return tuple(min(stable_hash(token, seed) for token in tokens)
                 for seed in range(num_perm))

def jaccard(left, right):
    return len(left & right) / max(1, len(left | right))

valid_a = [r for r in records if r.tenant_id == 'tenant-a' and not validate_schema(r)]
sets = {r.doc_id: shingles(decoded[r.doc_id]) for r in valid_a}
signatures = {doc_id: minhash_signature(tokens) for doc_id, tokens in sets.items()}
similarity = jaccard(sets['a-001'], sets['a-002'])
estimate = sum(a == b for a, b in zip(signatures['a-001'], signatures['a-002'])) / 32
assert similarity > 0.75
assert abs(similarity - estimate) < 0.25
print({'true_jaccard': round(similarity, 3), 'minhash_estimate': estimate})

## 8. 文档族隔离：先聚类，后切分

同一 exact group 必须属于一个 family；near-duplicate 超过阈值也合并。真实流水线还应利用 canonical URL、会话 ID、附件 hash、爬取重定向和人工标记。

切分函数只依赖 family_id 与 split_policy_version，因此增加无关样本不会搬动旧 family。时间外推评估则应按 family 的最早/最晚时间设计，不能在一个 family 内切开。

In [ ]:
class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)

doc_ids = [r.doc_id for r in valid_a]
uf = UnionFind(doc_ids)
for i, left in enumerate(doc_ids):
    for right in doc_ids[i+1:]:
        if exact_key('tenant-a', decoded[left]) == exact_key('tenant-a', decoded[right]):
            uf.union(left, right)
        elif jaccard(sets[left], sets[right]) >= 0.75:
            uf.union(left, right)

families = {doc_id: 'fam-' + hashlib.sha1(uf.find(doc_id).encode()).hexdigest()[:10]
            for doc_id in doc_ids}

def stable_split(family_id, policy='split-v1'):
    bucket = int(hashlib.sha256(f'{policy}:{family_id}'.encode()).hexdigest()[:8], 16) % 10
    return 'train' if bucket < 7 else ('valid' if bucket < 9 else 'test')

splits = {doc_id: stable_split(family) for doc_id, family in families.items()}
family_to_splits = defaultdict(set)
for doc_id, family in families.items():
    family_to_splits[family].add(splits[doc_id])

assert families['a-001'] == families['a-002']
assert families['a-003'] == families['a-004']
assert all(len(parts) == 1 for parts in family_to_splits.values())
print({'families': families, 'splits': splits})

## 9. 质量门禁不是一个总分

总分会掩盖致命问题。schema 失败、解码失败、跨租户混合、不可解释 offset 应硬拒绝；长度、PII、语言置信度可按用途路由到清洗、隔离或人工复核。

门禁输出 reason codes 而不是布尔值，便于按 source/tenant/rule_version 监控。训练集与线上输入应共用核心检查，但阈值可以因用途不同而版本化。

In [ ]:
def quality_gate(record):
    reasons = validate_schema(record)
    if reasons:
        return {'decision': 'reject', 'reasons': reasons}
    try:
        text, used_encoding = strict_decode(record.payload_bytes, record.declared_encoding)
    except UnicodeDecodeError:
        return {'decision': 'quarantine', 'reasons': ['decode_error']}
    try:
        normalized, mapping = normalize_view(text)
    except ValueError as exc:
        return {'decision': 'quarantine', 'reasons': [str(exc)]}
    language = coarse_language(normalized)
    _, pii = redact_for_log(text)
    soft = []
    if len(normalized) < 8:
        soft.append('too_short')
    if language['label'] == 'unknown':
        soft.append('language_unknown')
    if pii:
        soft.append('contains_pii')
    decision = 'review' if soft else 'accept'
    return {'decision': decision, 'reasons': soft, 'encoding': used_encoding,
            'language': language, 'char_count': len(normalized),
            'mapping_size': len(mapping), 'pii_count': len(pii)}

gate_results = {r.doc_id: quality_gate(r) for r in records}
assert gate_results['bad-001']['decision'] == 'reject'
assert gate_results['a-001']['decision'] == 'review'
assert 'contains_pii' in gate_results['a-001']['reasons']
print(json.dumps(gate_results, ensure_ascii=False, indent=2))

## 10. 漂移：监控输入分布，也监控规则输出

至少按 tenant/source 观察：长度分位数、语言比例、解码失败率、PII 命中率、拒绝原因、near-dup 比例、词表 OOV 与标签分布。均值稳定不代表尾部稳定。

下面用 Jensen-Shannon divergence 比较分类分布。样本太少时报警只代表“需要调查”，不是自动重训命令；阈值必须根据历史波动与业务损失标定。

In [ ]:
def distribution(values, vocabulary):
    counts = Counter(values)
    total = max(1, sum(counts.values()))
    return [counts.get(v, 0) / total for v in vocabulary]

def js_divergence(p, q, eps=1e-12):
    m = [(a+b)/2 for a, b in zip(p, q)]
    def kl(a, b):
        return sum(x * math.log((x+eps)/(y+eps), 2) for x, y in zip(a, b) if x > 0)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

vocab = ['zh', 'en', 'mixed', 'unknown']
baseline = distribution(['zh']*90 + ['en']*10, vocab)
current_ok = distribution(['zh']*85 + ['en']*15, vocab)
current_drift = distribution(['zh']*35 + ['en']*65, vocab)
assert js_divergence(baseline, current_drift) > js_divergence(baseline, current_ok)
print({'stable_js': round(js_divergence(baseline, current_ok), 4),
       'drift_js': round(js_divergence(baseline, current_drift), 4)})

## 11. 可回放且 tenant-scoped 的 manifest

manifest 不复制全文，而是冻结 tenant、输入快照、每个文档指纹、规则/切分/代码版本、UTC 构建时间和决策摘要。对象存储中的原始内容必须不可变或按版本寻址；构建时间是流水线参数，不在函数内部读取当前时钟，因此相同输入仍可重放。

普通 SHA-256 对手机号、短句等低熵内容可能被离线枚举，不能因为“不可逆”就视为匿名化。本例使用带密钥 BLAKE2 指纹，并把 fixture key 显式标为不可用于生产；线上密钥应由 KMS 管理、按用途轮换且不写入 manifest。manifest 必须按授权 tenant 构建，禁止在同一产物中混入另一个租户。

In [ ]:
FIXTURE_FINGERPRINT_KEY = b'fixture-only-key-never-use-in-production'
CODE_VERSION = 'quality-notebook-2026-07-28'
BUILD_TIME_UTC = '2026-07-28T00:00:00Z'

def content_fingerprint(payload, key):
    if not isinstance(key, bytes) or len(key) < 16:
        raise ValueError('fingerprint_key_too_short')
    return hashlib.blake2b(payload, key=key, digest_size=32).hexdigest()

def build_manifest(records, tenant_id, fingerprint_key, rule_version=RULE_VERSION,
                   code_version=CODE_VERSION, build_time_utc=BUILD_TIME_UTC):
    parsed_build_time = datetime.fromisoformat(build_time_utc.replace('Z', '+00:00'))
    if parsed_build_time.tzinfo is None or parsed_build_time.utcoffset() is None:
        raise ValueError('build_time_must_be_timezone_aware')
    scoped_records = [r for r in records if r.tenant_id == tenant_id]
    if not scoped_records:
        raise ValueError('empty_tenant_scope')
    entries = []
    for r in sorted(scoped_records, key=lambda item: item.doc_id):
        result = quality_gate(r)
        entries.append({
            'doc_id': r.doc_id,
            'content_keyed_blake2b': content_fingerprint(r.payload_bytes, fingerprint_key),
            'decision': result['decision'],
            'reasons': result['reasons'],
        })
    manifest = {
        'tenant_id': tenant_id,
        'schema_version': SCHEMA_VERSION,
        'rule_version': rule_version,
        'split_policy': 'split-v1',
        'code_version': code_version,
        'build_time_utc': build_time_utc,
        'input_snapshot': 'fixture-2026-07-01',
        'records': entries,
    }
    canonical = json.dumps(manifest, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    manifest['manifest_sha256'] = hashlib.sha256(canonical.encode()).hexdigest()
    return manifest

manifest1 = build_manifest(records, 'tenant-a', FIXTURE_FINGERPRINT_KEY)
manifest2 = build_manifest(list(reversed(records)), 'tenant-a', FIXTURE_FINGERPRINT_KEY)
manifest_b = build_manifest(records, 'tenant-b', FIXTURE_FINGERPRINT_KEY)
assert manifest1 == manifest2
assert manifest1['tenant_id'] == 'tenant-a' and manifest_b['tenant_id'] == 'tenant-b'
assert {row['doc_id'] for row in manifest1['records']} == {r.doc_id for r in records if r.tenant_id == 'tenant-a'}
assert all(row['doc_id'] != 'b-001' for row in manifest1['records'])
assert all('content_keyed_blake2b' in row and 'content_sha256' not in row
           for row in manifest1['records'])
assert len(manifest1['manifest_sha256']) == 64
print(json.dumps(manifest1, ensure_ascii=False, indent=2)[:1200])

## 12. 必须进入回归集的失败反例

- lower/分词规则把 E1042 拆成 E、1042，导致告警检索失败。
- 停用词表删除“不”，把“不要退款”变成“要退款”。
- errors=ignore 吞掉坏字节，样本看似成功但证据已损坏。
- 在随机切分后才去重，同一工单进入训练和测试。
- PII mask 改变长度，后续 offset 与审计位置错位。
- 全局 dedup hash 或跨 tenant manifest 泄露另一租户是否拥有同一文本。
- 对低熵内容只保存裸 SHA-256，攻击者可枚举候选内容比对摘要。
- 清洗直接覆盖 raw，之后无法重放旧规则或纠正误删。

In [ ]:
# 可执行合同测试：这些断言应与规则版本一起进入 CI。
raw = '退款未到账时，不要重复提交；错误码 Ｅ１０４２。'
view, mapping = normalize_view(raw)
assert '未' in view and '不要' in view
assert 'E1042' in view
assert len(view) == len(mapping)
assert project_clean_span(mapping, view.index('退款'), view.index('退款')+2) == (0, 2)
assert canonical_for_dedup('A  B') == canonical_for_dedup('Ａ B')
assert canonical_for_dedup('不能删除') != canonical_for_dedup('能删除')
assert strict_decode('中文'.encode('gb18030'), 'gb18030')[0] == '中文'
assert validate_schema(naive_time_record) == ['timezone_required']
try:
    strict_decode(b'\xff\xfe\xfa', 'utf-8')
    raise AssertionError('坏字节不应被静默接受')
except UnicodeDecodeError:
    pass
masked_mail, hits = redact_for_log('联系 x@y.cn，E1042 不能删除')
assert len(masked_mail) == len('联系 x@y.cn，E1042 不能删除')
assert hits[0]['type'] == 'email'
assert 'E1042' in masked_mail and '不能' in masked_mail
assert families['a-001'] == families['a-002']
assert all(len(value) == 1 for value in family_to_splits.values())
assert manifest1['rule_version'] == RULE_VERSION
assert manifest1['code_version'] == CODE_VERSION and manifest1['build_time_utc'] == BUILD_TIME_UTC
assert {row['doc_id'] for row in manifest1['records']}.isdisjoint({row['doc_id'] for row in manifest_b['records']})
assert quality_gate(records[-1])['decision'] == 'reject'
print('NLP 数据质量合同回归：全部通过')

## 13. 服务、版本与安全

建议批处理/流式接口返回 document_id、decision、reason_codes、derived_artifact_uri、rule_version、manifest_id、trace_id。幂等键可由 tenant_id + doc_id + payload hash + rule_version 构成；相同输入重试不产生第二份样本。

权限边界要覆盖原始库、隔离区、脱敏视图、manifest 与指标。指标标签不能含邮箱、手机号、正文或高基数 doc_id。删除请求必须传播到原始对象、派生数据集、搜索索引、训练缓存和 lineage 清单，同时保留合规允许的删除审计。

## 14. 生产替换点与容量设计

教学实现使用 O(n²) 两两 Jaccard，仅适合小数据。生产可采用：

- 流式 schema：Protobuf/Avro + registry，死信队列承接不可恢复输入。
- 编码/语言：经领域验证的探测器，保留置信度和原始字节。
- near dedup：MinHash LSH、SimHash 或向量候选 + 精确确认；按 tenant/source 分片。
- PII：规则、NER 与 DLP 服务组合，明确漏检/误杀标注集。
- 数据版本：不可变对象存储 + 表快照 + lineage catalog。
- 监控：按来源与语种分桶，shadow 运行新规则，差异审查后再切换。

复杂度上，exact hash 为 O(total bytes)；全量两两 near dedup 为 O(n²)，必须通过 LSH/分桶降候选；offset 映射占 O(text length)，可按需要压缩成连续区间。

## 15. 原始资料与进一步阅读

1. Unicode Consortium, Unicode Normalization Forms (UAX #15)：https://unicode.org/reports/tr15/
2. Andrei Broder, On the resemblance and containment of documents, 1997：https://doi.org/10.1109/SEQUEN.1997.666900
3. Manku, Jain, Das Sarma, Detecting Near-Duplicates for Web Crawling, WWW 2007：https://doi.org/10.1145/1242572.1242592
4. Python Unicode HOWTO（实现语义的官方文档）：https://docs.python.org/3/howto/unicode.html
5. NIST Privacy Framework：https://www.nist.gov/privacy-framework

结论边界：这里的规则、阈值和小样本只证明数据合同可执行；它们没有证明跨语种编码检测、PII 召回率或线上去重阈值已经达标。